# 🖼️ Image Restoration using GFPGAN — CelebA-HQ (Full Dataset)

**Dataset:** CelebA-HQ 256x256 (~30,000 images)

### What this notebook does:
1. Loads **all** CelebA-HQ 256x256 images
2. Degrades each (blur + noise + downscale)
3. Restores with **GFPGAN v1.3** — purpose-built blind face restorer
4. Saves results to disk progressively (memory-safe)
5. Computes PSNR & SSIM across the full dataset

### Setup:
- **Accelerator:** GPU T4 x2
- **Internet:** ON
- **Dataset:** badasstechie/celebahq-resized-256x256

> Time estimate: ~30k images x ~0.5s = ~4 hours on T4

In [ ]:
# CELL 1: Install + patch basicsr
import subprocess
subprocess.run(['pip', 'install', '-q', 'gfpgan', 'scikit-image', 'opencv-python-headless', 'tqdm'], check=True)
import glob as _g
matches = _g.glob('/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py') + \
          _g.glob('/opt/conda/lib/python*/site-packages/basicsr/data/degradations.py')
for path in matches:
    txt = open(path).read()
    if 'functional_tensor' in txt:
        open(path, 'w').write(txt.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'))
        print(f'Patched: {path}')
print('Done!')

In [ ]:
# CELL 2: Imports & GPU check
import os, glob, cv2, numpy as np, torch, json as _json
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU! Enable in Settings -> Accelerator -> GPU T4 x2')

In [ ]:
# CELL 3: Load ALL CelebA-HQ images
DATASET_PATH = '/kaggle/input/celebahq-resized-256x256/celeba_hq_256'
if not os.path.exists(DATASET_PATH):
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if 'celeba' in d.lower():
                DATASET_PATH = os.path.join(root, d); break
print(f'Dataset: {DATASET_PATH}')

all_images = sorted(
    glob.glob(os.path.join(DATASET_PATH, '*.jpg')) +
    glob.glob(os.path.join(DATASET_PATH, '*.png'))
)
print(f'Total images: {len(all_images)}')

# Change to a smaller number (e.g. 100) for a quick test run
NUM_SAMPLES = len(all_images)
sample_paths = all_images[:NUM_SAMPLES]
print(f'Will process: {NUM_SAMPLES} images')

# Preview first 4
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, path in zip(axes, sample_paths[:4]):
    ax.imshow(Image.open(path).convert('RGB'))
    ax.set_title(os.path.basename(path), fontsize=9)
    ax.axis('off')
plt.suptitle('First 4 Images (Preview)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# CELL 4: Degradation function
def degrade_image(img):
    d = img.copy().filter(ImageFilter.GaussianBlur(radius=2))
    arr = np.clip(np.array(d).astype(np.float32) + np.random.normal(0,15,np.array(d).shape), 0, 255).astype(np.uint8)
    d = Image.fromarray(arr)
    d = d.resize((64,64), Image.LANCZOS).resize((256,256), Image.NEAREST)
    return d

# Preview
gt0 = Image.open(sample_paths[0]).convert('RGB').resize((256,256), Image.LANCZOS)
dg0 = degrade_image(gt0)
fig, axes = plt.subplots(1, 2, figsize=(10,5))
axes[0].imshow(gt0); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(dg0); axes[1].set_title('Degraded'); axes[1].axis('off')
plt.suptitle('Degradation Preview', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print('Degradation preview done!')

In [ ]:
# CELL 5: Load GFPGAN
from gfpgan import GFPGANer
print('Loading GFPGAN v1.3...')
restorer = GFPGANer(
    model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
    upscale=1, arch='clean', channel_multiplier=2, bg_upsampler=None
)
print('GFPGAN loaded!')

In [ ]:
# CELL 6: Run restoration on full dataset (saves to disk, memory-safe)
OUT_DIR = '/kaggle/working/restored'
os.makedirs(OUT_DIR, exist_ok=True)

metrics_log = []
errors = []
SAVE_ORIGINALS = False  # set True to also save original & degraded copies

for i, path in enumerate(tqdm(sample_paths, desc='Restoring')):
    name = os.path.splitext(os.path.basename(path))[0]
    out_path = os.path.join(OUT_DIR, f'{name}_restored.png')
    if os.path.exists(out_path):  # resume-safe: skip already done
        continue
    try:
        gt  = Image.open(path).convert('RGB').resize((256,256), Image.LANCZOS)
        deg = degrade_image(gt)
        _, _, rbgr = restorer.enhance(
            cv2.cvtColor(np.array(deg), cv2.COLOR_RGB2BGR),
            has_aligned=False, only_center_face=False, paste_back=True)
        res = Image.fromarray(cv2.cvtColor(rbgr, cv2.COLOR_BGR2RGB))
        res.save(out_path)
        if SAVE_ORIGINALS:
            gt.save(os.path.join(OUT_DIR, f'{name}_original.png'))
            deg.save(os.path.join(OUT_DIR, f'{name}_degraded.png'))
        gt_a  = np.array(gt);  dg_a = np.array(deg);  rs_a = np.array(res)
        metrics_log.append({'name': name,
            'psnr_before': float(psnr(gt_a, dg_a, data_range=255)),
            'psnr_after':  float(psnr(gt_a, rs_a, data_range=255)),
            'ssim_before': float(ssim(gt_a, dg_a, data_range=255, channel_axis=2)),
            'ssim_after':  float(ssim(gt_a, rs_a, data_range=255, channel_axis=2))})
        if (i+1) % 500 == 0:
            g = np.mean([m['psnr_after']-m['psnr_before'] for m in metrics_log])
            print(f'[{i+1}/{NUM_SAMPLES}] Avg PSNR gain: {g:+.2f} dB')
    except Exception as e:
        errors.append({'name': name, 'error': str(e)})

with open('/kaggle/working/metrics.json', 'w') as f:
    _json.dump(metrics_log, f, indent=2)
print(f'Done! {len(metrics_log)} processed, {len(errors)} errors')

In [ ]:
# CELL 7: Visualize 4 random restored samples
import random
done = glob.glob(os.path.join(OUT_DIR, '*_restored.png'))
show = random.sample(done, min(4, len(done)))

fig, axes = plt.subplots(len(show), 3, figsize=(14, 5*len(show)))
if len(show) == 1: axes = [axes]

for row, rpath in enumerate(show):
    name = os.path.basename(rpath).replace('_restored.png', '')
    orig = next((p for p in sample_paths if os.path.splitext(os.path.basename(p))[0]==name), None)
    res  = Image.open(rpath).convert('RGB')
    gt   = Image.open(orig).convert('RGB').resize((256,256),Image.LANCZOS) if orig else res
    deg  = degrade_image(gt) if orig else res
    for col,(img,title) in enumerate(zip([gt,deg,res],['Ground Truth','Degraded','Restored (GFPGAN)'])):
        axes[row][col].imshow(img); axes[row][col].axis('off')
        if row==0: axes[row][col].set_title(title, fontsize=13, fontweight='bold')
    axes[row][0].set_ylabel(name, fontsize=9, rotation=0, labelpad=60, va='center')

plt.suptitle('Sample Results — GFPGAN v1.3', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/sample_results.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved sample_results.png')

In [ ]:
# CELL 8: Full dataset metrics summary
with open('/kaggle/working/metrics.json') as f:
    metrics_log = _json.load(f)

pb = [m['psnr_before'] for m in metrics_log]
pa = [m['psnr_after']  for m in metrics_log]
sb = [m['ssim_before'] for m in metrics_log]
sa = [m['ssim_after']  for m in metrics_log]
gp = [a-b for a,b in zip(pa,pb)]
gs = [a-b for a,b in zip(sa,sb)]

print('='*50)
print(f"{'Metric':<15} {'Before':>9} {'After':>9} {'Gain':>9}")
print('='*50)
print(f"{'PSNR (dB)':<15} {np.mean(pb):>9.2f} {np.mean(pa):>9.2f} {np.mean(gp):>+9.2f}")
print(f"{'SSIM':<15} {np.mean(sb):>9.3f} {np.mean(sa):>9.3f} {np.mean(gs):>+9.3f}")
print('='*50)
print(f'Total: {len(metrics_log)} images')
print(f'Improved: {sum(1 for g in gp if g>0)} ({100*sum(1 for g in gp if g>0)/len(gp):.1f}%)')

plt.figure(figsize=(10,4))
plt.hist(gp, bins=50, color='steelblue', edgecolor='white')
plt.axvline(0, color='red', linestyle='--', label='No change')
plt.axvline(np.mean(gp), color='green', linestyle='--', label=f'Mean: {np.mean(gp):+.2f} dB')
plt.xlabel('PSNR Gain (dB)'); plt.ylabel('Count')
plt.title('PSNR Gain Distribution — Full CelebA-HQ', fontsize=14, fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('/kaggle/working/psnr_distribution.png', dpi=150)
plt.show()

In [ ]:
# CELL 9: Save model weights (.pth)
save_path = '/kaggle/working/gfpgan_v1.3.pth'
torch.save(restorer.gfpgan.state_dict(), save_path)
print(f'Model saved -> {save_path}  ({os.path.getsize(save_path)/1e6:.1f} MB)')

In [ ]:
# CELL 10: Restore YOUR OWN image
#
# HOW TO UPLOAD ON KAGGLE:
# 1. kaggle.com/datasets -> New Dataset -> upload image -> Publish (Private ok)
# 2. In this notebook: '+ Add Input' -> search your dataset -> Add
# 3. Image will be at: /kaggle/input/<dataset-slug>/<filename.jpg>
# 4. Paste that path below

INPUT_IMAGE_PATH = ''  # <-- paste your path here, e.g. '/kaggle/input/my-photos/face.jpg'

# Auto-find if not set
if not INPUT_IMAGE_PATH or not os.path.exists(INPUT_IMAGE_PATH):
    candidates = [
        f for f in (
            glob.glob('/kaggle/input/**/*.jpg', recursive=True) +
            glob.glob('/kaggle/input/**/*.png', recursive=True) +
            glob.glob('/kaggle/input/**/*.jpeg', recursive=True)
        )
        if 'celeba' not in f.lower()
    ]
    if candidates:
        INPUT_IMAGE_PATH = candidates[0]
        print(f'Auto-found: {INPUT_IMAGE_PATH}')
    else:
        print('No custom image found. Please follow the upload steps above.')
        raise SystemExit('Upload your image first!')

print(f'Using: {INPUT_IMAGE_PATH}')
inp = Image.open(INPUT_IMAGE_PATH).convert('RGB')
print(f'Size: {inp.size}')

_, _, out_bgr = restorer.enhance(
    cv2.cvtColor(np.array(inp), cv2.COLOR_RGB2BGR),
    has_aligned=False, only_center_face=False, paste_back=True
)
out_img = Image.fromarray(cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB))

fig, axes = plt.subplots(1, 2, figsize=(12,6))
axes[0].imshow(inp);     axes[0].set_title('Your Input',           fontsize=13, fontweight='bold'); axes[0].axis('off')
axes[1].imshow(out_img); axes[1].set_title('Restored (GFPGAN v1.3)', fontsize=13, fontweight='bold'); axes[1].axis('off')
plt.suptitle('Custom Image Restoration', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

out_name = os.path.splitext(os.path.basename(INPUT_IMAGE_PATH))[0]
out_path = f'/kaggle/working/{out_name}_restored.png'
out_img.save(out_path)
print(f'Saved -> {out_path}')